# Grammar-KT full-v1 dataset

This notebook opens the newest frozen baseline dataset and exposes its five scientific objects: canonical **GrammarCells**, generator **K\***, the validated item bank, true **Q\***, and observable learner interactions.

GrammarCell, generator K\*, and downstream discovered K-hat hypotheses remain distinct. This notebook reads the public dataset only; it does not load private learner-oracle trajectories or downstream experiment outputs.

In [1]:
import os

DATA_FOLDER = os.environ.get("GRAMMAR_KT_DATA_FOLDER", "data/grammar_kt_full_v1")
DATA_FOLDER

'data/grammar_kt_full_v1'

In [2]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

cwd = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / "pyproject.toml").is_file()),
    cwd,
)
folder = Path(DATA_FOLDER).expanduser()
DATA_DIR = (folder if folder.is_absolute() else ROOT / folder).resolve()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

def read_json(relative_path):
    return json.loads((DATA_DIR / relative_path).read_text(encoding="utf-8"))

def read_jsonl(relative_path):
    return pd.read_json(DATA_DIR / relative_path, lines=True, compression="infer")

def show(frame, rows=None):
    visible = frame if rows is None else frame.head(rows)
    display(visible.reset_index(drop=True))

manifest = read_json("manifest.json")
assert manifest["dataset_id"] == "grammar_kt_full_v1"
assert manifest["status"] == "FROZEN_BASELINE_COMPLETE"

SETUP = {
    "repo_root": str(ROOT),
    "dataset": str(DATA_DIR),
    "dataset_id": manifest["dataset_id"],
    "status": manifest["status"],
    "read_only": True,
    "oracle_loaded": False,
}
SETUP

{'repo_root': '/home/abdullah/grammar_kt_harness',
 'dataset': '/home/abdullah/grammar_kt_harness/data/grammar_kt_full_v1',
 'dataset_id': 'grammar_kt_full_v1',
 'status': 'FROZEN_BASELINE_COMPLETE',
 'read_only': True,
 'oracle_loaded': False}

## 1. Frozen release overview

The manifest is the authoritative release summary. Acquisition uses seen grammar; probes are non-updating and cover the complete fixed item bank.

In [3]:
release_scale = pd.DataFrame(
    [{"object": name, "count": count} for name, count in manifest["scale"].items()]
)
stream = manifest["simulation"]["stream_summary"]
phase_summary = pd.DataFrame([
    {
        "phase": phase,
        "events": events,
        "correct": stream["correct_counts_by_phase"][phase],
        "accuracy": stream["correct_counts_by_phase"][phase] / events,
    }
    for phase, events in stream["phase_counts"].items()
])
regime_summary = pd.DataFrame([
    {"grammar_regime": regime, "events": events}
    for regime, events in stream["grammar_regime_counts"].items()
])

show(release_scale)
show(phase_summary)
show(regime_summary)

,object,count
0,source_relations,228
1,canonical_grammar_cells,75
2,generator_kcs,18
3,items,113
4,q_edges,269
5,learners,1000
6,interactions,283000


,phase,events,correct,accuracy
0,acquisition,170000,84438,0.496694
1,probe,113000,65986,0.583947


,grammar_regime,events
0,seen,254000
1,unseen_combination,20000
2,unseen_value,9000


## 2. Canonical GrammarCells

A GrammarCell records the linguistic structure instantiated by an item. It is not itself a learner KC. The complete 75-row table is retained as grammar_cells; the rendered view shows a compact sample.

In [4]:
cell_records = read_jsonl("grammar/cells.jsonl")
regime_records = read_jsonl("grammar/regime_assignments.jsonl")
feature_columns = ["tense", "aspect", "voice", "polarity", "clause", "modal"]

grammar_cells = pd.concat(
    [
        cell_records[["cell_id", "source_ids"]].reset_index(drop=True),
        pd.json_normalize(cell_records["features"])[feature_columns].reset_index(drop=True),
    ],
    axis=1,
).merge(
    regime_records[
        ["cell_id", "grammar_regime", "combination_subtype", "item_support", "selection_reason"]
    ],
    on="cell_id",
    validate="one_to_one",
)
grammar_cells = grammar_cells.sort_values(["grammar_regime", "cell_id"]).reset_index(drop=True)

cell_regime_counts = (
    grammar_cells.groupby("grammar_regime", as_index=False)
    .agg(cells=("cell_id", "size"), items=("item_support", "sum"))
)
show(cell_regime_counts)
show(
    grammar_cells[
        ["cell_id", "grammar_regime", "combination_subtype", *feature_columns, "item_support"]
    ],
    rows=12,
)

,grammar_regime,cells,items
0,seen,54,84
1,unseen_combination,15,20
2,unseen_value,6,9


,cell_id,grammar_regime,combination_subtype,tense,aspect,voice,polarity,clause,modal,item_support
0,gc_04a854582c08aa84,seen,None,NA,none,active,negative,imperative,none,2
1,gc_08d90a35b669ed28,seen,None,present,perfect,active,positive,polar_question,none,1
2,gc_16d9f6e33f0517ec,seen,None,NA,none,active,positive,declarative,will,1
3,gc_172c3f1039296750,seen,None,NA,none,active,positive,polar_question,would,1
4,gc_17fcb3b3c0156c98,seen,None,past,none,passive,negative,declarative,none,1
5,gc_19656e31c58c2c07,seen,None,present,none,passive,positive,declarative,none,2
6,gc_28d8bf2edc192d4c,seen,None,NA,none,active,negative,declarative,could,1
7,gc_2924f6e63e1c1da2,seen,None,NA,progressive,active,positive,declarative,should,1
8,gc_2d6eb4f93cba4c6b,seen,None,past,perfect,active,positive,declarative,none,1
9,gc_325e05b06bb38886,seen,None,NA,none,active,positive,polar_question,may,2


## 3. Generator KC inventory K\*

These 18 reusable English grammatical operations are the declared latent skills in the controlled simulator. They were frozen before responses were generated and are not discovered from those responses.

In [5]:
kc_records = read_jsonl("kcs.jsonl")
q_matrix = pd.read_csv(DATA_DIR / "q_matrix.csv")
q_item_support = q_matrix.drop(columns="item_id").sum(axis=0)

generator_kcs = pd.DataFrame([
    {
        "kc_id": row["id"],
        "name": row["name"],
        "family": row["family"],
        "activation_rule": json.dumps(row["activation_rule"], sort_keys=True),
        "cells": int(row["cell_support"]),
        "items": int(q_item_support[row["id"]]),
        "description": row["description"],
    }
    for row in kc_records.to_dict(orient="records")
]).sort_values("kc_id").reset_index(drop=True)

assert set(generator_kcs["kc_id"]) == set(q_matrix.columns) - {"item_id"}
show(generator_kcs)

,kc_id,name,family,activation_rule,cells,items,description
0,gkc_aspect_perfect,perfect construction,declared_operation,"{""cell"": {""aspect"": [""perfect"", ""perfect_progressive""]}}",30,42,Construct perfect HAVE and its participial dependency.
1,gkc_aspect_progressive,progressive construction,declared_operation,"{""cell"": {""aspect"": [""progressive"", ""perfect_progressive""]}}",17,29,Construct progressive BE and its -ing dependency.
2,gkc_be_passive,canonical BE-passive,declared_operation,"{""cell"": {""voice"": ""passive""}}",13,22,Construct passive BE and the lexical past participle.
3,gkc_finite_past,past finite-form selection,declared_operation,"{""cell"": {""tense"": ""past""}}",13,20,Select past finite morphology on the operator or main verb.
4,gkc_finite_present,present finite-form selection,declared_operation,"{""cell"": {""tense"": ""present""}}",17,30,Select present finite morphology and agreement conditions.
5,gkc_imperative,imperative clause formation,declared_operation,"{""cell"": {""clause"": ""imperative""}}",2,4,Realize an English imperative main clause.
6,gkc_modal_can,central modal CAN,dimension_value_operation,"{""cell"": {""modal"": ""can""}}",4,7,Select central modal CAN and its base-form complement.
7,gkc_modal_could,central modal COULD,dimension_value_operation,"{""cell"": {""modal"": ""could""}}",5,5,Select central modal COULD and its base-form complement.
8,gkc_modal_may,central modal MAY,dimension_value_operation,"{""cell"": {""modal"": ""may""}}",5,8,Select central modal MAY and its base-form complement.
9,gkc_modal_might,central modal MIGHT,dimension_value_operation,"{""cell"": {""modal"": ""might""}}",3,5,Select central modal MIGHT and its base-form complement.


## 4. Validated item bank and true Q-matrix Q\*

Each Q\* edge follows deterministically from a declared generator-KC activation rule. The full 113-row learner-facing bank is retained as item_bank, and the complete binary matrix as q_matrix.

In [6]:
item_records = read_jsonl("items/items.jsonl")
assert not q_matrix["item_id"].duplicated().any()
assert set(q_matrix["item_id"]) == set(item_records["item_id"])
q_active = (
    q_matrix.set_index("item_id")
    .apply(lambda row: [kc_id for kc_id, active in row.items() if int(active) == 1], axis=1)
    .rename("generator_kc_ids")
    .reset_index()
)

item_provenance = pd.DataFrame({
    "item_id": item_records["item_id"],
    "generation_campaign": [
        metadata.get("campaign", "baseline_n3")
        for metadata in item_records["generation_metadata"]
    ],
    "curation_rank": [metadata["rank"] for metadata in item_records["selection_metadata"]],
    "packaging_correction": [isinstance(value, dict) for value in item_records["correction_metadata"]],
})

item_bank = (
    item_records[
        ["item_id", "cell_id", "format", "prompt", "target_answer", "accepted_answers"]
    ]
    .merge(
        grammar_cells[["cell_id", "grammar_regime", *feature_columns]],
        on="cell_id",
        validate="many_to_one",
    )
    .merge(q_active, on="item_id", validate="one_to_one")
    .merge(item_provenance, on="item_id", validate="one_to_one")
    .sort_values(["grammar_regime", "cell_id", "item_id"])
    .reset_index(drop=True)
)

measurement = read_json("provenance/measurement/audit.json")
assert measurement["status"] == "PASS"
measurement_summary = pd.DataFrame([
    {"diagnostic": key, "value": value}
    for key, value in measurement["counts"].items()
] + [
    {
        "diagnostic": "identical_q_columns",
        "value": len(measurement["identifiability"]["identical_q_columns"]),
    },
    {
        "diagnostic": "near_identical_q_column_pairs",
        "value": len(measurement["identifiability"]["near_identical_q_column_pairs"]),
    },
])

show(measurement_summary)
show(
    item_bank[
        [
            "item_id", "cell_id", "grammar_regime", "generator_kc_ids", "format",
            "prompt", "target_answer", "accepted_answers", "generation_campaign",
        ]
    ],
    rows=15,
)
show(q_matrix, rows=10)

,diagnostic,value
0,canonical_cells,75
1,measured_cells,75
2,items,113
3,generator_kcs,18
4,q_edges,269
5,q_density,0.132252
6,q_rank,18
7,full_column_rank,True
8,distinct_canonical_cell_activation_rows,75
9,kc_pairs,153


,item_id,cell_id,grammar_regime,generator_kc_ids,format,prompt,target_answer,accepted_answers,generation_campaign
0,cue_bounded_imperative_gc_04a854582c08aa84_01,gc_04a854582c08aa84,seen,"[gkc_imperative, gkc_negation]",controlled_production,A child reaches toward a hot pan. Give a warning using an ordinary uncontracted negative imperative. Lexical cue chu...,Do not touch the hot pan.,[Do not touch the hot pan],cue_bounded_imperative_production_v1
1,cue_bounded_imperative_gc_04a854582c08aa84_02,gc_04a854582c08aa84,seen,"[gkc_imperative, gkc_negation]",controlled_production,"The paint is still wet, so warn your friend. Write an ordinary uncontracted negative imperative. Cues (not in the ta...",Do not touch the wet paint.,[Do not touch the wet paint],cue_bounded_imperative_production_v1
2,candidate_gc_08d90a35b669ed28_02,gc_08d90a35b669ed28,seen,"[gkc_aspect_perfect, gkc_finite_present, gkc_polar_question]",controlled_production,"Leo started his homework after dinner. Ask whether it is complete now, using “Leo” and “finish his homework”: [____]",Has Leo finished his homework?,"[Has Leo finished his homework?, Has Leo finished his homework]",baseline_n3
3,unchanged_rescue_gc_16d9f6e33f0517ec_01,gc_16d9f6e33f0517ec,seen,[gkc_modal_will],controlled_production,"Ben has promised to help after lunch. Complete the sentence using the cue ""carry"": Ben ___ the bags to the car.",Ben will carry the bags to the car.,[will carry],unchanged_prompt_zero_coverage_rescue_v1
4,candidate_gc_172c3f1039296750_01,gc_172c3f1039296750,seen,"[gkc_modal_would, gkc_polar_question]",controlled_production,The room is hot. Ask politely if your friend is willing to open the window. Use: she / open / the window\nResponse: ...,Would she open the window?,[Would she open the window?],baseline_n3
5,candidate_gc_17fcb3b3c0156c98_03,gc_17fcb3b3c0156c98,seen,"[gkc_be_passive, gkc_finite_past, gkc_negation]",controlled_production,"The shop was closed yesterday, so the bread ___ (deliver).",The bread was not delivered.,"[was not delivered, wasn't delivered]",baseline_n3
6,candidate_gc_19656e31c58c2c07_01,gc_19656e31c58c2c07,seen,"[gkc_be_passive, gkc_finite_present]",controlled_production,"Before every class, a worker gets the chairs ready in rows. Complete the sentence using “arrange”: The chairs ____ i...",The chairs are arranged in rows before every class.,[are arranged],baseline_n3
7,candidate_gc_19656e31c58c2c07_02,gc_19656e31c58c2c07,seen,"[gkc_be_passive, gkc_finite_present]",controlled_production,The shop sells all the bread before noon each day. Complete the sentence using the cue (sell): The bread ____ before...,The bread is sold before noon each day.,[is sold],baseline_n3
8,determinacy_intervention_gc_28d8bf2edc192d4c_02,gc_28d8bf2edc192d4c,seen,"[gkc_modal_could, gkc_negation]",controlled_production,"Mia's hand was hurt, so lifting the heavy box was impossible for her. Complete the sentence using a negative declara...",Mia could not lift the heavy box.,"[could not lift, couldn't lift]",explicit_construction_determinacy_intervention_v1
9,determinacy_intervention_gc_2924f6e63e1c1da2_02,gc_2924f6e63e1c1da2,seen,"[gkc_aspect_progressive, gkc_modal_should]",controlled_production,"The dishes are dirty, and Mia needs to clean them now. Use a positive declarative clause with “should” in the progre...",Mia should be washing the dishes.,[Mia should be washing the dishes.],explicit_construction_determinacy_intervention_v1


,item_id,gkc_aspect_perfect,gkc_aspect_progressive,gkc_be_passive,gkc_finite_past,gkc_finite_present,gkc_imperative,gkc_modal_can,gkc_modal_could,gkc_modal_may,gkc_modal_might,gkc_modal_must,gkc_modal_shall,gkc_modal_should,gkc_modal_will,gkc_modal_would,gkc_negation,gkc_non_subject_wh_question,gkc_polar_question
0,candidate_gc_0397fa37f2228649_01,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,candidate_gc_0397fa37f2228649_02,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,candidate_gc_08d90a35b669ed28_02,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
3,candidate_gc_0bbfaece19d0255d_03,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0
4,candidate_gc_172c3f1039296750_01,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
5,candidate_gc_17fcb3b3c0156c98_03,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
6,candidate_gc_19656e31c58c2c07_01,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
7,candidate_gc_19656e31c58c2c07_02,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8,candidate_gc_19ed2b72505b3a96_01,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
9,candidate_gc_19ed2b72505b3a96_02,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


## 5. Observable learner interactions

The public stream contains only dataset-neutral observable fields. It has 170,000 acquisition events followed by one non-updating probe of every bank item for each learner. The full table is retained as interactions; the rendered output is summarized and sampled.

In [7]:
interactions = read_jsonl("interactions.jsonl.gz")
observable_schema = manifest["simulation"]["stream_summary"]["schemas"]["observable"]
assert list(interactions.columns) == observable_schema
assert len(interactions) == manifest["scale"]["interactions"]
assert interactions["learner_id"].nunique() == manifest["scale"]["learners"]

interaction_summary = (
    interactions.groupby(["phase", "grammar_regime"], as_index=False)
    .agg(
        events=("correct", "size"),
        learners=("learner_id", "nunique"),
        accuracy=("correct", "mean"),
    )
)
response_sample = interactions.head(15).merge(
    item_bank[["item_id", "cell_id", "generator_kc_ids", "prompt", "target_answer"]],
    on="item_id",
    validate="many_to_one",
)

show(interaction_summary)
show(
    response_sample[
        [
            "learner_id", "sequence_index", "phase", "grammar_regime", "item_id",
            "cell_id", "generator_kc_ids", "prompt", "target_answer", "correct",
        ]
    ]
)

,phase,grammar_regime,events,learners,accuracy
0,acquisition,seen,170000,1000,0.496694
1,probe,seen,84000,1000,0.586952
2,probe,unseen_combination,20000,1000,0.574600
3,probe,unseen_value,9000,1000,0.576667


,learner_id,sequence_index,phase,grammar_regime,item_id,cell_id,generator_kc_ids,prompt,target_answer,correct
0,learner_000001,1,acquisition,seen,candidate_gc_4601bed02c004e37_01,gc_4601bed02c004e37,[gkc_finite_past],"Yesterday, Mia needed some milk, so she ___ to the shop. (walk)",Mia walked to the shop.,1
1,learner_000001,2,acquisition,seen,candidate_gc_e7fef77abc10b5ba_01,gc_e7fef77abc10b5ba,"[gkc_aspect_perfect, gkc_modal_would, gkc_negation]","Mia had a map, so she found the house. Without the map, [____]. (find)",she would not have found the house,1
2,learner_000001,3,acquisition,seen,candidate_gc_a0fe7953b9fffca8_03,gc_a0fe7953b9fffca8,"[gkc_be_passive, gkc_finite_present, gkc_negation]","The cleaner washes the office windows on Fridays, but leaves the kitchen windows dirty. Complete using “clean”: The ...",The kitchen windows are not cleaned on Fridays.,0
3,learner_000001,4,acquisition,seen,candidate_gc_8b7e8b31b496e102_02,gc_8b7e8b31b496e102,"[gkc_aspect_progressive, gkc_be_passive, gkc_finite_present, gkc_negation]","The workers are painting the front door, but they have not started the back door. Complete the sentence using the ve...",The back door is not being painted.,1
4,learner_000001,5,acquisition,seen,unchanged_rescue_gc_4a4c9d34c1b5bce4_01,gc_4a4c9d34c1b5bce4,"[gkc_aspect_perfect, gkc_modal_must]","The kitchen light is off, but Ben left it on when he went out. Complete the sentence using the cue in brackets: Ben ...",Ben must have turned it off.,0
5,learner_000001,6,acquisition,seen,candidate_gc_c2cf343c43ce329d_02,gc_c2cf343c43ce329d,"[gkc_aspect_perfect, gkc_finite_present, gkc_negation]","Maya planned to wash the car today, but it is still dirty. Complete the sentence using the cue (wash): Maya ____ the...",Maya has not washed the car.,0
6,learner_000001,7,acquisition,seen,determinacy_intervention_gc_a94168b2255fe062_01,gc_a94168b2255fe062,"[gkc_aspect_progressive, gkc_modal_will, gkc_negation]",Mia's car will be at the repair shop tomorrow. Use a negative active declarative with will and the progressive aspec...,Mia will not be driving to work.,1
7,learner_000001,8,acquisition,seen,candidate_gc_dfd385cd88abc20f_02,gc_dfd385cd88abc20f,"[gkc_be_passive, gkc_modal_must, gkc_negation]",The wet floor is dangerous. Complete the rule: The door ___ until the floor is dry.,The door must not be opened until the floor is dry.,1
8,learner_000001,9,acquisition,seen,determinacy_intervention_gc_a94168b2255fe062_02,gc_a94168b2255fe062,"[gkc_aspect_progressive, gkc_modal_will, gkc_negation]",Mia has a dentist appointment tomorrow morning. Complete the sentence using the negative progressive with will and t...,Mia will not be working.,0
9,learner_000001,10,acquisition,seen,candidate_gc_172c3f1039296750_01,gc_172c3f1039296750,"[gkc_modal_would, gkc_polar_question]",The room is hot. Ask politely if your friend is willing to open the window. Use: she / open / the window\nResponse: ...,Would she open the window?,1


## 6. Integrity and scientific boundary

The hashes below are recomputed from the files opened by this notebook. Ordinary dataset use does not require the private oracle file.

In [8]:
core_artifacts = [
    "grammar/cells.jsonl",
    "kcs.jsonl",
    "items/items.jsonl",
    "q_matrix.csv",
    "interactions.jsonl.gz",
]
integrity_rows = []
for relative_path in core_artifacts:
    path = DATA_DIR / relative_path
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    expected = manifest["artifact_inventory"][relative_path]["sha256"]
    integrity_rows.append({
        "artifact": relative_path,
        "bytes": path.stat().st_size,
        "sha256": observed,
        "matches_manifest": observed == expected,
    })

scientific_boundary = pd.DataFrame([
    {"contract": key, "value": value}
    for key, value in manifest["scientific_boundary"].items()
])
show(pd.DataFrame(integrity_rows))
show(scientific_boundary)

FINAL_DATASET_SUMMARY = {
    "dataset_id": manifest["dataset_id"],
    "status": manifest["status"],
    "grammar_cells": len(grammar_cells),
    "generator_kcs": len(generator_kcs),
    "items": len(item_bank),
    "q_edges": int(q_matrix.drop(columns="item_id").to_numpy().sum()),
    "learners": interactions["learner_id"].nunique(),
    "interactions": len(interactions),
    "all_core_hashes_match": all(row["matches_manifest"] for row in integrity_rows),
    "scientific_distinction": "GrammarCell != generator K* != discovered K_hat",
    "oracle_loaded": False,
}
FINAL_DATASET_SUMMARY

,artifact,bytes,sha256,matches_manifest
0,grammar/cells.jsonl,22345,a6bf8041e93fdea21d0ba7b318a24cb64610a6064517a2f7fbbbf3aca2fa4e1a,True
1,kcs.jsonl,15814,f3b308dcda7c11886e1dc67476dbe6adbcb53b8676b643c62d226fe68cfe0355,True
2,items/items.jsonl,81382,46e2fcbd723d11f6b30ecdd197fb05c39f04f7b0fc8f61e2183ffb8fb7110011,True
3,q_matrix.csv,8404,b6df582478f05976ceb200da6edc2b31fb305da64498e5ddb5f473a9459bf5bf,True
4,interactions.jsonl.gz,2158088,9272ca86a647e3b13c9ce52b5381dde215f7ef448e4a19a41a22495fa99ef97f,True


,contract,value
0,generator_kcs_frozen_before_responses,True
1,q_star_frozen_before_responses,True
2,item_bank_frozen_before_responses,True
3,learner_outcomes_used_to_construct_k_star_or_q_star,False
4,discovered_kcs_or_kt_results_read,False
5,private_oracle_required_for_ordinary_kt,False


{'dataset_id': 'grammar_kt_full_v1',
 'status': 'FROZEN_BASELINE_COMPLETE',
 'grammar_cells': 75,
 'generator_kcs': 18,
 'items': 113,
 'q_edges': 269,
 'learners': 1000,
 'interactions': 283000,
 'all_core_hashes_match': True,
 'scientific_distinction': 'GrammarCell != generator K* != discovered K_hat',
 'oracle_loaded': False}

## Interpretation boundary

K\*, Q\*, mastery, and simulator parameters are controlled synthetic truth—not claims about human cognitive structure. Automatic item validation is not human pedagogical gold, and this English instantiation does not establish cross-lingual validity.